In [ ]:
!pip install efficientnet_pytorch

In [ ]:
import os
from sklearn.metrics import f1_score
import torch
from torch import nn, optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from efficientnet_pytorch import EfficientNet
from tqdm import tqdm

In [ ]:
DATA_DIR = os.path.join(PROJECT_PATH, "data/Yummly-66K/food_min/region_split_images")
BATCH_SIZE = 64
NUM_EPOCHS = 15
LR = 1e-4
MODEL_NAME = "efficientnet-b4"
SAVE_PATH = os.path.join(PROJECT_PATH, "models", "efficientnet_region_custom.pth")
USE_AMP = True   # Mixed precision

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(380),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_tf = transforms.Compose([
    transforms.Resize(420),
    transforms.CenterCrop(380),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = datasets.ImageFolder(os.path.join(DATA_DIR, "train"), train_tf)
test_dataset   = datasets.ImageFolder(os.path.join(DATA_DIR, "test"), test_tf)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=12, pin_memory=True, persistent_workers=True)
test_loader   = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=12, pin_memory=True, persistent_workers=True)

num_classes = len(train_dataset.classes)
print("Classes:", train_dataset.classes)

In [ ]:
model = EfficientNet.from_pretrained(MODEL_NAME)

in_features = model._fc.in_features
model._fc = nn.Linear(in_features, num_classes)  # Replace final layer

# Prompt: Freeze the first two layers of my netowork
# Model: ChatGPT based off GPT5.1


# Freezing two blocks
for idx, block in enumerate(model._blocks):
    if idx < 2:  # freeze blocks 0 and 1
        for param in block.parameters():
            param.requires_grad = False

model = model.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

In [ ]:
def train_one_epoch():
    model.train()
    running_loss = 0
    running_correct = 0

    for imgs, labels in tqdm(train_loader, desc="Training"):
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            outputs = model(imgs)
            loss = criterion(outputs, labels)

        preds = outputs.argmax(dim=1)
        running_correct += (preds == labels).sum().item()
        running_loss += loss.item() * imgs.size(0)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()



    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = running_correct / len(train_dataset)
    return epoch_loss, epoch_acc


def validate():
    model.eval()
    running_loss = 0
    running_correct = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for imgs, labels in tqdm(test_loader, desc="Validation"):
            imgs, labels = imgs.to(device), labels.to(device)

            outputs = model(imgs)
            loss = criterion(outputs, labels)

            preds = outputs.argmax(dim=1)
            running_correct += (preds == labels).sum().item()
            running_loss += loss.item() * imgs.size(0)

            # For F1
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())


    epoch_loss = running_loss / len(test_dataset)
    epoch_acc = running_correct / len(test_dataset)
    epoch_f1_macro = f1_score(all_labels, all_preds, average="macro")

    return epoch_loss, epoch_acc, epoch_f1_macro

In [ ]:
# Prompt: Write me a training loop that trains my model over multiple epochs using my training and validation functions and saves the best model
# Model: ChatGPT based off GPT5.1

best_acc = 0.0
best_acc_f1 = 0.0

for epoch in range(NUM_EPOCHS):
    print(f"\n----- Epoch {epoch+1}/{NUM_EPOCHS} -----")

    train_loss, train_acc = train_one_epoch()
    test_loss, test_acc, test_f1 = validate()

    scheduler.step()

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Test  Loss: {test_loss:.4f} | Test   Acc: {test_acc:.4f}")

    # save best model
    if test_acc > best_acc:
        best_acc = test_acc
        best_acc_f1 = test_f1
        torch.save(model.state_dict(), SAVE_PATH)
        print(f"Saved new best model → {SAVE_PATH}")

print("Training complete.")
print(f"Best test accuracy: {best_acc:.4f}")
print(f"Associated f1-score: {best_acc_f1:.4f}")